# Machine Learning - Practical 08: Gradient Descent and a Small MLP

**Course:** Machine Learning - National University of Kyiv-Mohyla Academy (NaUKMA)

**Instructor:** Dmytro Kuzmenko - kuzmenko@ukma.edu.ua

| | |
|---|---|
| Week | 8 |
| Module | 8. Optimization and Neural Networks |
| Format | Practical session (not graded - exam preparation) |
| Estimated time | 1.5-2 h of active work including discussion |
| Prerequisites | P01-P07; basic Python; linear algebra (gradient, matrix product) |


## AI Use Disclosure

Fill this in before submitting (see course policy).

| Field | Your entry |
|---|---|
| AI tools used | |
| Nature of assistance | |
| Representative prompts or relevant interaction | |
| What I independently verified or changed | |

## Learning Objectives

- Implement manual gradient descent on a convex quadratic and characterize the learning-rate regimes (too small, good, too large, diverging).
- Train a small multilayer perceptron in PyTorch on make_moons with a manual training loop, and read the loss curve.
- Compare full-batch and minibatch training and interpret the noise in the loss curve.

## Warm-up (10 min)

### Question 1 (multiple choice)

Gradient descent on a smooth loss currently uses a learning rate that is 10x the value that worked well in a previous run. What is the most likely outcome?

- A. The loss converges a bit faster, because a larger learning rate always means faster progress.
- B. The loss oscillates or diverges: each step overshoots the minimum, and the update may even increase the loss.
- C. Nothing changes; the learning rate only affects the first iteration.
- D. The model becomes more accurate on the training set but worse on the validation set.


**Your answer:**

### Question 2 (quick reasoning)

The gradient of a loss points in the direction of the steepest *increase* of the loss. Why do we update the parameters in the opposite direction, and what role does the learning rate play?


**Your answer:**

### Question 3 (quick reasoning)

A network with two linear layers and no activation function between them is equivalent to a single linear layer. Why does that make non-linear activations necessary in deep networks?


**Your answer:**

## Guided Exercise (60 min)

We first study gradient descent on a toy quadratic where the exact gradient is known, then train a small neural network with PyTorch. All experiments are deterministic (seeds fixed).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
%matplotlib inline
np.random.seed(42)
_ = torch.manual_seed(42)

### Task 1: Manual gradient descent on a 2-D quadratic

**Experiment.** Minimize f(w) = 0.5 * (w1^2 + 5 * w2^2), starting from w = (2.5, -2.0), with plain gradient descent w <- w - lr * grad(f)(w). Run 60 iterations for four learning rates: 0.005 (too small), 0.05 (good), 0.30 (too large), 0.42 (diverging).

In [2]:
def gd_quadratic(lr, w0=np.array([2.5, -2.0]), iters=60, a=5.0):
    w = w0.copy()
    trajectory = [w.copy()]
    losses = []
    for _ in range(iters):
        grad = np.array([w[0], a * w[1]])     # analytic gradient of 0.5*(w1^2 + a*w2^2)
        w = w - lr * grad
        trajectory.append(w.copy())
        losses.append(0.5 * (w[0] ** 2 + a * w[1] ** 2))
    return np.array(trajectory), np.array(losses)

learning_rates = [0.005, 0.05, 0.30, 0.42]
trajectories = {}
loss_curves = {}
for lr in learning_rates:
    traj, losses = gd_quadratic(lr)
    trajectories[lr] = traj
    loss_curves[lr] = losses
    print("lr = {:.3f}: final loss = {:.4f}".format(lr, losses[-1]))

lr = 0.005: final loss = 2.1917
lr = 0.050: final loss = 0.0066
lr = 0.300: final loss = 0.0000
lr = 0.420: final loss = 927090.6882


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# loss curves (log scale)
for lr in learning_rates:
    axes[0].plot(loss_curves[lr], label="lr = {}".format(lr))
axes[0].set_yscale("log")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("loss f(w)")
axes[0].set_title("Loss curves for different learning rates")
axes[0].legend()
axes[0].set_ylim(1e-6, 1e4)
# contour plot + trajectories for two rates
w1 = np.linspace(-3, 3, 300)
w2 = np.linspace(-3, 3, 300)
W1, W2 = np.meshgrid(w1, w2)
F = 0.5 * (W1 ** 2 + 5 * W2 ** 2)
axes[1].contour(W1, W2, F, levels=40, cmap="viridis")
for lr in [0.05, 0.30]:
    traj = trajectories[lr]
    axes[1].plot(traj[:, 0], traj[:, 1], marker=".", ms=3, label="lr = {}".format(lr))
axes[1].set_xlabel("w1"); axes[1].set_ylabel("w2")
axes[1].set_title("Gradient descent trajectories on the quadratic")
axes[1].legend()
plt.tight_layout()

**Decision.** Which learning rate would you use for this problem, and what is your criterion (speed, stability, final loss)?


**Your answer:**

**Evidence.** Point to the specific features of the loss curves that distinguish the four regimes.


**Your answer:**

**Interpretation.** For this quadratic, the update in the w2 coordinate is w2 <- (1 - 5*lr) * w2. Use this formula to explain why lr = 0.30 converges with a zigzag while lr = 0.42 diverges.


**Your answer:**

**Counterfactual.** The stability boundary is |1 - 5*lr| < 1, i.e., lr < 0.4. Predict what the trajectory looks like at exactly lr = 0.4, and what happens if we change the coefficient 5 to 20 (a steeper direction).


**Your answer:**

### Task 2: A small MLP on make_moons (PyTorch, manual loop)

**Experiment.** Two-moon classification with 300 points (noise 0.15). Train an MLP with two hidden layers of 16 units (tanh) and one output neuron with logits, using BCEWithLogitsLoss and SGD (lr = 0.1, 400 epochs). Record the loss and training accuracy every epoch, then plot the loss curve and the decision boundary.

In [4]:
X, y = make_moons(n_samples=300, noise=0.15, random_state=42)
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16), nn.Tanh(),
            nn.Linear(16, 16), nn.Tanh(),
            nn.Linear(16, 1),
        )
    def forward(self, x):
        return self.net(x)

def train_mlp(lr, epochs=400):
    torch.manual_seed(42)
    model = MLP()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    losses, accs = [], []
    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(X_t)
        loss = loss_fn(logits, y_t)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        with torch.no_grad():
            pred = (torch.sigmoid(logits) > 0.5).float()
            accs.append((pred == y_t).float().mean().item())
    return model, losses, accs

model_good, losses_good, accs_good = train_mlp(lr=0.1)
print("final loss: {:.4f}   final train accuracy: {:.3f}".format(
    losses_good[-1], accs_good[-1]))

final loss: 0.2702   final train accuracy: 0.887


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses_good, lw=1.5)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE loss")
axes[0].set_title("Training loss (full batch, lr = 0.1)")
# decision boundary
xx, yy = np.meshgrid(np.linspace(-1.6, 2.6, 220), np.linspace(-1.2, 1.6, 220))
grid = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)
with torch.no_grad():
    zz = torch.sigmoid(model_good(grid)).numpy().reshape(xx.shape)
axes[1].contourf(xx, yy, zz, levels=20, cmap="RdBu_r", alpha=0.6)
axes[1].contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=1)
axes[1].scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", s=12, edgecolors="k", lw=0.3)
axes[1].set_title("Decision boundary of the trained MLP")
plt.tight_layout()

**Evidence.** Describe the shape of the loss curve: does it decrease smoothly, and where does it flatten? What final accuracy is reached?


**Your answer:**

**Interpretation.** Why is the decision boundary curved rather than linear? Which part of the network is responsible for that (activations, hidden units, number of layers)?


**Your answer:**

**Counterfactual.** Train the same network with lr = 1.0 and with lr = 0.01 and compare the loss curves with the lr = 0.1 run. Which one oscillates or diverges, and which one is too slow?


In [6]:
model_high, losses_high, _ = train_mlp(lr=1.0)
model_low, losses_low, _ = train_mlp(lr=0.01)
plt.figure(figsize=(7, 3.5))
plt.plot(losses_good, label="lr = 0.1")
plt.plot(losses_high, label="lr = 1.0", alpha=0.8)
plt.plot(losses_low, label="lr = 0.01", alpha=0.8)
plt.xlabel("epoch"); plt.ylabel("BCE loss")
plt.title("Loss curves: learning-rate counterfactual")
plt.legend()
plt.ylim(0, 2.0)
plt.tight_layout()

**Interpretation.** Interpret the three curves: which learning rate is in the "too large" regime for this network, and which in the "too small" regime? What visual signature do you use?


**Your answer:**

## Discussion Questions (15 min)

1. On the quadratic, a too-large lr diverges cleanly; on the MLP, lr = 1.0 oscillates but does not explode. What property of the loss landscape (curvature, boundedness) explains the difference?
2. A flat training loss that stays high can mean several things: lr too small, architecture too weak, or a bug. Which diagnostic (loss curve shape, train accuracy, gradient norms) helps you tell them apart?
3. Minibatch SGD uses a noisy gradient estimate. Why is the noise sometimes helpful (escaping flat regions) and sometimes harmful (slow final convergence)?
4. Why do we use BCEWithLogitsLoss (logits + sigmoid inside the loss) instead of applying sigmoid first and then MSE? What numerical property makes the former preferable?
5. tanh vs ReLU: what happens to the gradient of tanh for large |z|, and how does that affect deep networks (vanishing gradients)?
6. If the validation loss stops improving while the training loss keeps dropping, is the fix a different learning rate, more data, or regularization? Justify.

## Challenge (25 min)

### Task 3: Add minibatching and compare with full-batch training

The training loop above used the whole dataset per step (batch = full). Rewrite it to use minibatches of size 32: shuffle indices every epoch, split into chunks of 32, and take one optimizer step per chunk. Train for the same number of epochs (400) with the same lr = 0.1, store the per-epoch average loss in `losses_mini`, then plot `losses_mini` together with `losses_good` (full batch) and interpret the difference in noise and convergence speed.

In [7]:
# Your code

**Interpretation.** Compare the two curves: which is noisier, which converges faster in the first epochs, and which reaches a lower final loss? Explain the noise in terms of gradient estimation.


**Your answer:**

**Counterfactual.** Predict what changes if the batch size is reduced to 8 or increased to 128. What happens in the limit batch = full (your baseline) and batch = 1 (pure SGD)?


**Your answer:**

## Takeaways

- Gradient descent steps against the gradient; the learning rate controls the step size and has three regimes: too small (slow), good (fast convergence), too large (oscillation or divergence).
- The stability boundary is set by the curvature of the loss: for a quadratic with curvature a, convergence requires lr < 2 / a.
- A neural network with non-linear activations can fit curved boundaries; the loss curve is the primary signal for debugging training.
- PyTorch training loops follow a fixed pattern: forward pass, loss, backward, optimizer step; seeds must be fixed for reproducibility.
- Minibatch SGD trades exact gradients for cheap noisy ones: noisier curves, often faster early progress, and a final loss that fluctuates around the minimum.
- Diagnose training problems with the loss curve plus accuracy/gradient evidence before changing architecture or data.